# Semana 6 · Del perceptrón a la red multicapa

**Nivel Intermedio** · Semana 6 · Clase expositiva [A] · 2 horas · CPU (no hace falta GPU)

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/melvinpqbsc/Alto_Rendimiento_IA/blob/main/Nivel_Intermedio/clases/sem06_perceptron_mlp.ipynb)

**Objetivos.** Al terminar la clase podrás:

1. Explicar qué calcula un perceptrón, entrenarlo con su regla de aprendizaje y decir qué problemas no puede resolver (XOR).
2. Hacer en papel la propagación hacia adelante (*forward pass*) de una red pequeña, y después escribirla con matrices.
3. Comparar las funciones de activación sigmoide, tanh y ReLU, y decidir cuál va en cada lugar de una red.
4. Explicar por qué sin no linealidad una red de muchas capas equivale a una sola, y verlo en datos.

**Prerrequisitos.** La clase `Linear` y `Sequential` (semana 1), matriz × vector (semana 2), derivadas, regla de la cadena y *softmax* (semana 4), regresión logística y validación (semana 5). Todavía **no** usamos PyTorch (semana 8) ni explicamos cómo se entrena una red (semana 7): hoy los pesos los ponemos nosotros, y cuando hace falta entrenar, `fit` de sklearn lo hace por dentro.

**Agenda** (después de la apertura de 15 minutos, donde se publica la tabla de posiciones del Sprint 0).

| Parte | Tema | Minutos |
|---|---|---|
| 1 | Una neurona: el perceptrón | 10 |
| 2 | Aprender de los errores: la regla del perceptrón | 15 |
| 3 | El muro: XOR | 10 |
| 4 | Dos capas: XOR a mano y la propagación en papel | 20 |
| 5 | La red en matrices | 15 |
| 6 | Funciones de activación | 15 |
| 7 | Por qué importa la no linealidad | 15 |
| 8 | Cierre y tareas para la semana 7 | 5 |

Ejecuta esta celda primero: carga las librerías y los datos, fija la semilla y define `check`, la función que revisa tus ejercicios de "Tu turno".

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.datasets import load_digits
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression, Perceptron
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

SEED = 0
rng = np.random.default_rng(SEED)

URL = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv"
COLS = ["bill_depth_mm", "flipper_length_mm"]
penguins = pd.read_csv(URL).dropna(subset=COLS + ["species"])
penguins = penguins[penguins["species"].isin(["Adelie", "Gentoo"])]

COLORS = ["tab:blue", "tab:orange"]


def check(name, test):
    """Ejecuta una prueba y muestra ✓, ✗ o "todavía sin resolver", sin detener el notebook."""
    try:
        test()
        print(f"✓ {name}")
    except NotImplementedError:
        print(f"… {name}: todavía sin resolver")
    except AssertionError as e:
        print(f"✗ {name}: {e}")
    except Exception as e:
        print(f"✗ {name}: {type(e).__name__}: {e}")


def plot_points(ax, X, y, names=("clase 0", "clase 1"), size=60):
    """Dibuja los puntos de las dos clases, con su leyenda."""
    for c in (0, 1):
        ax.scatter(X[y == c, 0], X[y == c, 1], s=size, color=COLORS[c], edgecolor="white", label=names[c], zorder=3)


def plot_line(ax, w, b, xlim, **kwargs):
    """Dibuja la recta w[0]·x + w[1]·y + b = 0 dentro del rango xlim (w[1] no puede ser 0)."""
    xs = np.array(xlim)
    ax.plot(xs, -(w[0] * xs + b) / w[1], **kwargs)


def plot_regions(ax, predict, X, y, margin=0.5, size=14):
    """Pinta la región de cada clase según predict(puntos) y encima los puntos."""
    xs = np.linspace(X[:, 0].min() - margin, X[:, 0].max() + margin, 300)
    ys = np.linspace(X[:, 1].min() - margin, X[:, 1].max() + margin, 300)
    gx, gy = np.meshgrid(xs, ys)
    pred = predict(np.c_[gx.ravel(), gy.ravel()]).reshape(gx.shape)
    ax.contourf(gx, gy, pred, levels=[-0.5, 0.5, 1.5], colors=COLORS, alpha=0.2)
    plot_points(ax, X, y, size=size)


print(len(penguins), "pingüinos Adelie y Gentoo")

## 1. Una neurona: el perceptrón

Imagina que decides si vas a la playa. Miras tres cosas, y no todas pesan lo mismo:

| Evidencia | ¿Hoy? ($x$) | Peso ($w$) |
|---|---|---|
| Hace sol | 1 | $+3$ |
| Van tus amigos | 1 | $+2$ |
| Mañana hay examen | 1 | $-4$ |

Sumas cada evidencia por su peso y agregas tu ánimo de partida, $b = -0.5$ (eres un poco perezoso): $3 + 2 - 4 - 0.5 = 0.5$. Como es mayor que cero, **vas**.

Eso es un **perceptrón** (*perceptron*), propuesto por Frank Rosenblatt en 1958: una suma con pesos seguida de una decisión de sí o no.

$$\hat{y} = \begin{cases} 1 & \text{si } w \cdot x + b > 0 \\ 0 & \text{si no} \end{cases}$$

Los $w$ son los **pesos** (*weights*) y $b$ es el **sesgo** (*bias*); no tiene nada que ver con el sesgo de sesgo-varianza de la semana 5, solo comparten el nombre. La función que decide sí o no se llama **escalón** (*step*).

In [ ]:
def step(z):
    """El escalón: 1 si z > 0, si no 0."""
    return (z > 0).astype(int)


def perceptron(x, w, b):
    """La decisión de un perceptrón. x puede ser un vector o un lote con forma (n, d)."""
    return step(x @ w + b)


beach_w = np.array([3.0, 2.0, -4.0])
beach_b = -0.5
print("sol, amigos y examen:", perceptron(np.array([1, 1, 1]), beach_w, beach_b))
print("solo sol y examen:   ", perceptron(np.array([1, 0, 1]), beach_w, beach_b))
print("solo sol:            ", perceptron(np.array([1, 0, 0]), beach_w, beach_b))

### Compuertas lógicas

Con pesos bien elegidos, un perceptrón calcula funciones lógicas. La compuerta **AND** da 1 solo si las dos entradas son 1; la compuerta **OR** da 1 si alguna lo es. Con dos entradas hay cuatro casos, y la igualdad $w_1 x_1 + w_2 x_2 + b = 0$ es una **recta** que los separa: de un lado el perceptrón dice 1, del otro dice 0.

In [ ]:
GATE_X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
AND_Y = np.array([0, 0, 0, 1])
OR_Y = np.array([0, 1, 1, 1])

and_w, and_b = np.array([1.0, 1.0]), -1.5
or_w, or_b = np.array([1.0, 1.0]), -0.5

print("x1 x2  AND  OR")
for x, a, o in zip(GATE_X, perceptron(GATE_X, and_w, and_b), perceptron(GATE_X, or_w, or_b)):
    print(f" {x[0]}  {x[1]}   {a}    {o}")

fig, axes = plt.subplots(1, 2, figsize=(8, 3.6), sharey=True)
for ax, y, w, b, title in [(axes[0], AND_Y, and_w, and_b, "AND"), (axes[1], OR_Y, or_w, or_b, "OR")]:
    plot_points(ax, GATE_X, y, names=("da 0", "da 1"), size=120)
    plot_line(ax, w, b, (-0.5, 1.5), color="gray", linestyle="--", label="la recta del perceptrón")
    ax.set(title=title, xlabel="$x_1$", xlim=(-0.5, 1.5), ylim=(-0.5, 1.5))
axes[0].set_ylabel("$x_2$")
fig.legend(*axes[0].get_legend_handles_labels(), loc="lower center", ncol=3, fontsize=9)
fig.subplots_adjust(bottom=0.25)
plt.show()

### Tu turno 1: elige los pesos

Encuentra pesos para dos compuertas nuevas. Hay muchas respuestas correctas; prueba con lápiz antes de ejecutar.

1. **NOR**: da 1 solo si las dos entradas son 0. Pesos `nor_w` (dos números) y sesgo `nor_b`.
2. **Mayoría de tres**: tres entradas, da 1 si al menos dos son 1. Pesos `maj_w` (tres números) y sesgo `maj_b`.

In [ ]:
nor_w, nor_b = None, None  # TODO
maj_w, maj_b = None, None  # TODO

In [ ]:
def test_nor():
    if nor_w is None:
        raise NotImplementedError
    pred = perceptron(GATE_X, np.asarray(nor_w), nor_b)
    assert (pred == [1, 0, 0, 0]).all(), f"para (0,0), (0,1), (1,0), (1,1) tu perceptrón da {pred.tolist()}"


def test_maj():
    if maj_w is None:
        raise NotImplementedError
    X3 = np.array([[a, b, c] for a in (0, 1) for b in (0, 1) for c in (0, 1)])
    pred = perceptron(X3, np.asarray(maj_w), maj_b)
    assert (pred == (X3.sum(axis=1) >= 2)).all(), f"falla en {X3[pred != (X3.sum(axis=1) >= 2)].tolist()}"


check("NOR", test_nor)
check("mayoría de tres", test_maj)

## 2. Aprender de los errores: la regla del perceptrón

Elegir pesos a mano no escala. Rosenblatt propuso que la neurona los aprenda de sus errores, como quien ajusta una receta probando: si salió sin sal, le pones más a lo que estaba en la olla.

Miramos los ejemplos uno por uno. Si el perceptrón **acierta**, no se toca nada. Si **se equivoca**, se mueven los pesos de las entradas que estaban encendidas:

$$w \leftarrow w + \eta\,(y - \hat{y})\,x \qquad\qquad b \leftarrow b + \eta\,(y - \hat{y})$$

- Dijo 0 y era 1: $y - \hat{y} = +1$, los pesos suben.
- Dijo 1 y era 0: $y - \hat{y} = -1$, los pesos bajan.

$\eta$ es la tasa de aprendizaje (semana 4). Sigamos la regla en AND, empezando con todo en cero. Una **época** (*epoch*) es una pasada completa por los datos.

In [ ]:
w, b = np.zeros(2), 0.0
snapshots = []
print("época  x       y  dijo    w después   b después")
for epoch in range(1, 11):
    mistakes = 0
    for x, target in zip(GATE_X, AND_Y):
        pred = int(x @ w + b > 0)
        if pred != target:
            w = w + (target - pred) * x
            b = b + (target - pred)
            mistakes += 1
            print(f"{epoch:>3}    {x}   {target}   {pred}     {w}    {b:>5}")
    snapshots.append((w.copy(), b))
    if mistakes == 0:
        print(f"época {epoch}: ningún error, se detiene")
        break
print("resultado:", perceptron(GATE_X, w, b), " AND:", AND_Y)

La recta se mueve con cada época hasta dejar el punto (1, 1) solo de su lado. El punto (1, 0) queda justo sobre la recta final: como la regla pide $z > 0$, cuenta como 0.

In [ ]:
fig, ax = plt.subplots(figsize=(4.6, 4))
plot_points(ax, GATE_X, AND_Y, names=("da 0", "da 1"), size=120)
for i, (sw, sb) in enumerate(snapshots):
    if sw[1] != 0:  # una recta vertical no se puede dibujar como y = f(x)
        final = i == len(snapshots) - 1
        plot_line(ax, sw, sb, (-0.5, 1.5), color="black" if final else "gray", alpha=1 if final else 0.35,
                  label="la recta final" if final else None)
ax.set(xlabel="$x_1$", ylabel="$x_2$", xlim=(-0.5, 1.5), ylim=(-0.5, 1.5), title="La recta, época tras época")
ax.legend(loc="lower left", fontsize=8)
plt.show()

Esta regla tiene una garantía fuerte, el **teorema de convergencia del perceptrón** (demostrado a comienzos de los años 60): **si existe una recta (o un plano) que separa las dos clases, la regla encuentra una en un número finito de pasos.**

### Tu turno 2: el perceptrón completo

Completa `perceptron_fit(X, y, epochs, lr)`, con el mismo bucle de arriba pero para cualquier cantidad de columnas:

- Empieza con `w = np.zeros(X.shape[1])` y `b = 0.0`.
- En cada época recorre los ejemplos, y si se equivoca aplica la regla con `lr` como $\eta$.
- Guarda en la lista `errors` cuántos errores hubo en cada época. Si una época no tuvo ninguno, termina antes.
- Devuelve `w, b, errors`.

La prueba lo usa con dos medidas estandarizadas de los pingüinos: profundidad del pico y largo de la aleta.

In [ ]:
def perceptron_fit(X, y, epochs=20, lr=1.0):
    # TODO
    raise NotImplementedError

In [ ]:
X_peng = StandardScaler().fit_transform(penguins[COLS].to_numpy(dtype=float))
y_peng = (penguins["species"] == "Gentoo").to_numpy(dtype=int)


def test_fit_and():
    w, b, errors = perceptron_fit(GATE_X, AND_Y, epochs=20)
    assert (perceptron(GATE_X, w, b) == AND_Y).all(), "no aprendió AND"
    assert errors[-1] == 0, "la última época debería terminar sin errores"


def test_fit_stops():
    w, b, errors = perceptron_fit(GATE_X, AND_Y, epochs=100)
    assert len(errors) < 100, "debe terminar antes cuando una época no tiene errores"


def test_fit_penguins():
    w, b, errors = perceptron_fit(X_peng, y_peng, epochs=50)
    assert w.shape == (2,), f"w debería tener forma (2,), tiene {w.shape}"
    assert (perceptron(X_peng, w, b) == y_peng).all(), "debería separar a los dos grupos sin errores"


check("aprende AND", test_fit_and)
check("termina antes si no hay errores", test_fit_stops)
check("separa a los pingüinos", test_fit_penguins)

Y la llamada de librería. `sklearn.linear_model.Perceptron` implementa esta misma regla. Como siempre, lo comparamos con una línea base.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    penguins[COLS].to_numpy(dtype=float), y_peng, test_size=0.25, stratify=y_peng, random_state=SEED)

base = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
scaler = StandardScaler().fit(X_train)
clf = Perceptron(random_state=SEED).fit(scaler.transform(X_train), y_train)
print(f"línea base: accuracy {base.score(X_test, y_test):.3f}")
print(f"perceptrón: accuracy {clf.score(scaler.transform(X_test), y_test):.3f}   w = {clf.coef_[0].round(2)}, b = {clf.intercept_[0]:.2f}")

X_std = scaler.transform(penguins[COLS].to_numpy(dtype=float))
fig, ax = plt.subplots(figsize=(5.5, 4))
plot_points(ax, X_std, y_peng, names=("Adelie", "Gentoo"), size=22)
plot_line(ax, clf.coef_[0], clf.intercept_[0], (X_std[:, 0].min(), X_std[:, 0].max()), color="gray", linestyle="--", label="la recta aprendida")
ax.set(xlabel="profundidad del pico (estandarizada)", ylabel="largo de la aleta (estandarizado)",
       ylim=(X_std[:, 1].min() - 0.3, X_std[:, 1].max() + 0.3))
ax.legend(fontsize=8)
plt.show()

## 3. El muro: XOR

Piensa en las escaleras de una casa, con un interruptor abajo y otro arriba. Si los dos están iguales, la luz está apagada; si están distintos, encendida. Eso es la compuerta **XOR**: da 1 cuando **exactamente una** de las entradas es 1.

| $x_1$ | $x_2$ | XOR |
|---|---|---|
| 0 | 0 | 0 |
| 0 | 1 | 1 |
| 1 | 0 | 1 |
| 1 | 1 | 0 |

Los dos puntos de cada clase están en **esquinas opuestas** de un cuadrado. Probemos la regla, que si existiera una recta la encontraría, y también 200 000 rectas al azar:

In [ ]:
XOR_Y = np.array([0, 1, 1, 0])

xor_clf = Perceptron(eta0=1.0, shuffle=False, penalty=None, random_state=SEED)
print("época  aciertos de 4")
for epoch in range(1, 9):
    xor_clf.partial_fit(GATE_X, XOR_Y, classes=[0, 1])
    print(f"{epoch:>3}    {(xor_clf.predict(GATE_X) == XOR_Y).sum()}")

n_lines = 200_000
random_w = rng.normal(size=(n_lines, 2))
random_b = rng.normal(size=n_lines)
pred = (GATE_X @ random_w.T + random_b > 0).T          # (200000, 4): lo que dice cada recta
hits = (pred == XOR_Y).sum(axis=1)                     # (200000,): los aciertos de cada recta
print("\nde 200 000 rectas al azar, cuántas logran cada número de aciertos:")
for k, count in zip(*np.unique(hits, return_counts=True)):
    print(f"  {k} de 4: {count} rectas")

fig, ax = plt.subplots(figsize=(4, 3.6))
plot_points(ax, GATE_X, XOR_Y, names=("da 0", "da 1"), size=120)
ax.set(title="XOR: ninguna recta los separa", xlabel="$x_1$", ylabel="$x_2$", xlim=(-0.5, 1.5), ylim=(-0.5, 1.5))
ax.legend(loc="center", fontsize=8)
plt.show()

Ninguna llega a 4 de 4, y no es mala suerte. Las cuatro filas de la tabla piden a la vez:

$$\underbrace{b \le 0}_{(0,0)} \qquad \underbrace{w_2 + b > 0}_{(0,1)} \qquad \underbrace{w_1 + b > 0}_{(1,0)} \qquad \underbrace{w_1 + w_2 + b \le 0}_{(1,1)}$$

Si sumas las dos del medio, $w_1 + w_2 + 2b > 0$. Si sumas la primera y la última, $w_1 + w_2 + 2b \le 0$. Las dos cosas no pueden ser ciertas: **no existe ninguna recta**.

En 1969, Marvin Minsky y Seymour Papert publicaron *Perceptrons*, un libro con los límites de estas redes de una capa, incluido este. Eso contribuyó a que el entusiasmo por las redes neuronales cayera durante años. La salida ya se sospechaba: poner una neurona después de otra. Lo que faltaba era cómo entrenarlas, y eso lo vemos la semana que viene.

## 4. Dos capas: XOR a mano

Toma una hoja con los cuatro puntos de XOR dibujados. Con un solo corte recto de tijera no puedes separar los colores. Pero si **doblas la hoja** para que las dos esquinas iguales queden una encima de la otra, un solo corte basta. Una capa oculta hace eso: cambia la forma de los datos para que una recta alcance al final.

XOR se escribe con compuertas que ya sabemos hacer: "alguna está encendida **y** no están las dos". Es decir, `XOR = AND(OR, NAND)`, donde NAND es lo contrario de AND:

In [ ]:
nand_w, nand_b = np.array([-1.0, -1.0]), 1.5


def xor_forward(x):
    """Dos capas de perceptrones, con los pesos elegidos a mano."""
    h1 = perceptron(x, or_w, or_b)
    h2 = perceptron(x, nand_w, nand_b)
    H = np.stack([h1, h2], axis=-1)
    return perceptron(H, and_w, and_b), H


out, H = xor_forward(GATE_X)
print("x1 x2 | h1=OR h2=NAND | salida=AND | XOR")
for x, h, o, target in zip(GATE_X, H, out, XOR_Y):
    print(f" {x[0]}  {x[1]} |   {h[0]}     {h[1]}    |     {o}      |  {target}")

Mira la columna del medio. Las entradas (0, 1) y (1, 0), que estaban en esquinas opuestas, **caen en el mismo punto**, (1, 1): la hoja doblada. En este espacio nuevo, una recta sí alcanza.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8.4, 3.8))
plot_points(axes[0], GATE_X, XOR_Y, names=("da 0", "da 1"), size=120)
axes[0].set(title="Antes: la entrada", xlabel="$x_1$", ylabel="$x_2$", xlim=(-0.5, 1.5), ylim=(-0.5, 1.5))
axes[0].legend(loc="center", fontsize=8)

plot_points(axes[1], H, XOR_Y, names=("da 0", "da 1"), size=120)
plot_line(axes[1], and_w, and_b, (-0.5, 1.5), color="gray", linestyle="--", label="la recta de AND")
axes[1].annotate("(0,1) y (1,0)\naquí juntos", (1, 1), (0.6, 1.27), arrowprops=dict(arrowstyle="->"), fontsize=9)
axes[1].set(title="Después: la capa oculta", xlabel="$h_1$ (OR)", ylabel="$h_2$ (NAND)", xlim=(-0.5, 1.5), ylim=(-0.5, 1.5))
axes[1].legend(loc="lower left", fontsize=8)
plt.show()

Acabamos de armar un **perceptrón multicapa** (*multilayer perceptron*, **MLP**). La capa del medio se llama **capa oculta** (*hidden layer*) porque nadie le dice qué valores debe tomar: no aparece ni en la entrada ni en la respuesta.

### La propagación hacia adelante, en papel

Calcular la salida capa por capa, de izquierda a derecha, es la **propagación hacia adelante** (*forward pass*). Hagámosla con una red de 2 entradas, 2 neuronas ocultas y 1 salida. Las neuronas ocultas usan ReLU, que deja pasar lo positivo y convierte lo negativo en 0: $\text{ReLU}(z) = \max(z, 0)$ (la vemos en la parte 6). Con la entrada $x = (2, -1)$:

| Neurona | Suma con pesos | Resultado |
|---|---|---|
| $h_1$ | $1 \cdot 2 + 2 \cdot (-1) + 1 = 1$ | $\text{ReLU}(1) = 1$ |
| $h_2$ | $-1 \cdot 2 + 1 \cdot (-1) + 0 = -3$ | $\text{ReLU}(-3) = 0$ |
| salida | $2 \cdot 1 + (-1) \cdot 0 - 0.5 = 1.5$ | $\sigma(1.5) = 0.818$ |

Al final aplicamos la sigmoide $\sigma(z) = 1/(1 + e^{-z})$ de la regresión logística (semana 5), para leer el resultado como la probabilidad de la clase 1. **Esta es la cuenta que una red hace millones de veces.** Compruébala en código:

In [ ]:
def relu(z):
    return np.maximum(z, 0)


def sigmoid(z):
    return 1 / (1 + np.exp(-z))


x = np.array([2.0, -1.0])
h1 = relu(1 * x[0] + 2 * x[1] + 1)
h2 = relu(-1 * x[0] + 1 * x[1] + 0)
z = 2 * h1 + (-1) * h2 - 0.5
print(f"h1 = {h1}, h2 = {h2}, z = {z}, salida = {sigmoid(z):.3f}")

### Tu turno 3: una red en papel

Saca papel y lápiz. **Hazlo a mano, sin ejecutar nada.** Otra red con la misma forma, y la entrada $x = (1, 2)$:

$$h_1 = \text{ReLU}(1 \cdot x_1 - 2 \cdot x_2 + 1) \qquad h_2 = \text{ReLU}(-1 \cdot x_1 + 3 \cdot x_2 + 0)$$

$$z = 1 \cdot h_1 - 0.5 \cdot h_2 + 2 \qquad \text{salida} = \sigma(z)$$

Escribe aquí los tres números que te dieron: $h_1$, $h_2$ y $z$. ¿La red dice clase 0 o clase 1? (Recuerda que la sigmoide pasa de 0.5 cuando $z > 0$.) ¿Qué le pasó a $h_1$?

In [ ]:
paper_h1 = None  # TODO
paper_h2 = None  # TODO
paper_z = None   # TODO

In [ ]:
def test_paper():
    if paper_h1 is None or paper_h2 is None or paper_z is None:
        raise NotImplementedError
    x = np.array([1.0, 2.0])
    h1, h2 = relu(x[0] - 2 * x[1] + 1), relu(-x[0] + 3 * x[1])
    assert np.isclose(paper_h1, h1), "revisa h1: primero la suma, después ReLU"
    assert np.isclose(paper_h2, h2), "revisa h2: primero la suma, después ReLU"
    assert np.isclose(paper_z, h1 - 0.5 * h2 + 2), "revisa z: usa los h que ya pasaron por ReLU"


check("la red en papel", test_paper)

## 5. La red en matrices

Escribir cada neurona a mano no sirve para redes grandes. En la semana 2 vimos que **matriz × vector = una capa**. Juntemos los pesos de la red de papel, $h_1$ en la primera fila y $h_2$ en la segunda:

$$W_1 = \begin{pmatrix} 1 & 2 \\ -1 & 1 \end{pmatrix} \quad b_1 = \begin{pmatrix} 1 \\ 0 \end{pmatrix} \qquad\qquad W_2 = \begin{pmatrix} 2 & -1 \end{pmatrix} \quad b_2 = -0.5$$

$$h = \text{ReLU}(W_1 x + b_1) \qquad\qquad z = W_2 h + b_2$$

Cada capa es lo mismo: una suma con pesos y una función aplicada a cada número. Y como en la semana 2, con un lote (*batch*) de $n$ filas se escribe `X @ W.T`:

In [ ]:
W1 = np.array([[1.0, 2.0], [-1.0, 1.0]])
b1 = np.array([1.0, 0.0])
W2 = np.array([[2.0, -1.0]])
b2 = np.array([-0.5])

X = np.array([[2.0, -1.0], [0.0, 0.0], [1.0, 1.0], [-3.0, 2.0]])  # un lote de 4 entradas
H = relu(X @ W1.T + b1)
Z = H @ W2.T + b2
print("la primera fila es la cuenta en papel:")
print(f"X {X.shape} → H {H.shape} → Z {Z.shape}")
print("H =", H.tolist())
print("Z =", Z.ravel().tolist())
print("probabilidades:", sigmoid(Z).ravel().round(3))

En la semana 1 escribimos estas mismas piezas como clases, y son las que usa `nn.Module` en la semana 8. Las repetimos aquí tal cual:

In [ ]:
class Module:
    def __call__(self, x):
        return self.forward(x)

    def forward(self, x):
        raise NotImplementedError(f"{type(self).__name__} debe definir forward")

    def parameters(self):
        return []


class Linear(Module):
    def __init__(self, in_features, out_features, rng):
        self.weight = rng.normal(size=(out_features, in_features)) / np.sqrt(in_features)
        self.bias = np.zeros(out_features)

    def forward(self, x):
        return x @ self.weight.T + self.bias

    def parameters(self):
        return [self.weight, self.bias]


class ReLU(Module):
    def forward(self, x):
        return np.maximum(x, 0)


class Sequential(Module):
    def __init__(self, *layers):
        self.layers = list(layers)

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]


def softmax(Z):
    """Softmax de cada fila de Z (semana 4)."""
    E = np.exp(Z - Z.max(axis=1, keepdims=True))
    return E / E.sum(axis=1, keepdims=True)


digits = load_digits()
X_dig_train, X_dig_test, y_dig_train, y_dig_test = train_test_split(
    digits.data / 16, digits.target, test_size=0.25, stratify=digits.target, random_state=SEED)

net = Sequential(Linear(64, 32, rng), ReLU(), Linear(32, 10, rng))
scores = net(X_dig_test)
print("entrada:", X_dig_test.shape, "→ puntajes:", scores.shape, "→ probabilidades:", softmax(scores).shape)
print("formas de los parámetros:", [p.shape for p in net.parameters()])

baseline_dig = DummyClassifier(strategy="most_frequent").fit(X_dig_train, y_dig_train).score(X_dig_test, y_dig_test)
random_acc = np.mean(scores.argmax(axis=1) == y_dig_test)
print(f"línea base (la clase más común): accuracy {baseline_dig:.3f}")
print(f"red con pesos al azar:           accuracy {random_acc:.3f}")

La red funciona, en el sentido de que las formas calzan, pero con pesos al azar no sabe nada: acierta más o menos lo mismo que adivinar. **Tiene la forma correcta y los números equivocados.** Encontrar los números es entrenar.

Una capa de $a$ entradas y $c$ salidas tiene $a \cdot c$ pesos y $c$ sesgos. Una red es esa cuenta, capa por capa.

### Tu turno 4: la red sin clases

1. Completa `mlp_forward(X, weights, biases)`, que recibe una lista de matrices y una lista de sesgos. Aplica cada capa con `X @ W.T + b` y ReLU después de cada capa **menos la última**, que devuelve puntajes sin activación.
2. Completa `count_mlp_params(sizes)`, que recibe los tamaños de las capas, como `[64, 32, 10]`, y devuelve el total de parámetros. Calcula a mano el de `[64, 32, 10]` antes de probar.

In [ ]:
def mlp_forward(X, weights, biases):
    # TODO
    raise NotImplementedError


def count_mlp_params(sizes):
    # TODO
    raise NotImplementedError

In [ ]:
def test_forward():
    out = mlp_forward(X, [W1, W2], [b1, b2])
    assert out.shape == (4, 1), f"la salida debería ser (4, 1), es {out.shape}"
    assert np.allclose(out, Z), "no coincide con la cuenta H y Z de arriba"


def test_forward_net():
    layers = [net.layers[0], net.layers[2]]
    out = mlp_forward(X_dig_test, [l.weight for l in layers], [l.bias for l in layers])
    assert np.allclose(out, scores), "no coincide con Sequential"


def test_count():
    assert count_mlp_params([2, 2, 1]) == 9, "la red de papel tiene 9 parámetros"
    assert count_mlp_params([64, 32, 10]) == sum(p.size for p in net.parameters()), "no coincide con net"
    assert count_mlp_params([3, 1]) == 4, "una sola capa de 3 a 1 tiene 3 pesos y 1 sesgo"


check("mlp_forward con la red de papel", test_forward)
check("mlp_forward coincide con Sequential", test_forward_net)
check("count_mlp_params", test_count)

## 6. Funciones de activación

La función que va después de la suma con pesos se llama **función de activación** (*activation function*). El perceptrón usa el escalón, un interruptor de pared: apagado o encendido. El problema es que, para entrenar, el gradiente tiene que decirnos hacia dónde mover cada peso (semana 4), y **el escalón es plano**: su derivada es 0 en todas partes menos en el salto, donde no existe. Mover un peso un poquito no cambia nada, y la pendiente no da ninguna pista.

Por eso se usan funciones con pendiente:

- **Sigmoide** (*sigmoid*), $\sigma(z) = \dfrac{1}{1 + e^{-z}}$: un regulador de luz, de 0 a 1, en vez de un interruptor.
- **Tanh**, $\tanh(z) = \dfrac{e^{z} - e^{-z}}{e^{z} + e^{-z}}$: la misma forma, de −1 a 1 y centrada en 0.
- **ReLU** (*rectified linear unit*), $\text{ReLU}(z) = \max(z, 0)$: una válvula de una sola vía, que deja pasar lo positivo y bloquea lo negativo.

In [ ]:
def numerical_derivative(f, x, h=1e-5):
    """Diferencia centrada (semana 4)."""
    return (f(x + h) - f(x - h)) / (2 * h)


z = np.linspace(-5, 5, 501)
activations = [("escalón", step), ("sigmoide", sigmoid), ("tanh", np.tanh), ("ReLU", relu)]

fig, axes = plt.subplots(2, 4, figsize=(11, 4.6), sharex=True)
for col, (name, f) in enumerate(activations):
    axes[0, col].plot(z, f(z), color="tab:blue", linewidth=2)
    axes[0, col].set_title(name)
    axes[0, col].set_ylim(-1.2, 2.2 if name == "ReLU" else 1.2)
    slope = numerical_derivative(f, z)
    slope[np.abs(slope) > 2] = np.nan  # en el salto del escalón la derivada no existe
    axes[1, col].plot(z, slope, color="tab:orange", linewidth=2)
    axes[1, col].set_ylim(-0.1, 1.2)
    axes[1, col].set_xlabel("z")
    for ax in axes[:, col]:
        ax.axhline(0, color="lightgray", linewidth=1, zorder=0)
        ax.axvline(0, color="lightgray", linewidth=1, zorder=0)
axes[0, 0].set_ylabel("la función")
axes[1, 0].set_ylabel("su derivada")
plt.tight_layout()
plt.show()

for value in (0.0, 2.0, 5.0, -5.0):
    print(f"z = {value:>4}:  sigmoide' = {numerical_derivative(sigmoid, value):.4f}   "
          f"tanh' = {numerical_derivative(np.tanh, value):.4f}   ReLU' = {numerical_derivative(relu, value):.4f}")

Mira la fila de abajo. (El 0.5 de ReLU en $z = 0$ es un artefacto de la diferencia centrada: justo en el doblez, ReLU no tiene derivada.) La derivada de la sigmoide llega como mucho a 0.25, y lejos del centro casi desaparece: la neurona está **saturada**. Eso importa por la regla de la cadena de la semana 4: el gradiente que llega a las primeras capas es un **producto de derivadas**, una por capa.

In [ ]:
print("capas   sigmoide (z = 0)   sigmoide (z = 2)   ReLU (z > 0)")
for n in (1, 5, 10, 20):
    print(f"{n:>4}   {0.25**n:>16.2e}   {numerical_derivative(sigmoid, 2.0)**n:>16.2e}   {1.0**n:>12.0f}")

Con 10 sigmoides, en el **mejor** caso y sin contar los pesos, el gradiente se multiplica por menos de una millonésima: las primeras capas casi no se enteran de nada. Es el **gradiente que se desvanece** (*vanishing gradient*), y es una de las razones por las que hoy casi todas las redes usan ReLU en las capas ocultas. ReLU no es gratis: si una neurona queda siempre del lado negativo, su salida y su derivada son 0 y deja de aprender (una **neurona muerta**). Tu $h_1$ del Tu turno 3 estaba apagada para esa entrada; una neurona muerta está apagada para todas.

| | Salida | Derivada máxima | A favor | En contra |
|---|---|---|---|---|
| Escalón | 0 o 1 | 0 | decisión clara | no se puede entrenar con gradiente |
| Sigmoide | $(0, 1)$ | 0.25 | se lee como probabilidad | se satura, no centrada en 0 |
| Tanh | $(-1, 1)$ | 1 | centrada en 0 | se satura |
| ReLU | $[0, \infty)$ | 1 | rápida, no se satura hacia arriba | neuronas muertas |

**Dónde va cada una:** en las capas ocultas, ReLU es la opción por defecto. En la última capa, depende del problema: nada para una regresión, sigmoide para dos clases, *softmax* para varias (semana 4).

### Tu turno 5: tanh desde cero

Completa las dos funciones **usando solo `np.exp`**, sin `np.tanh`:

1. `my_tanh(z)`, con la fórmula de arriba.
2. `my_tanh_derivative(z)`. Dato útil: la derivada de tanh se puede escribir como $1 - \tanh^2(z)$.

In [ ]:
def my_tanh(z):
    # TODO
    raise NotImplementedError


def my_tanh_derivative(z):
    # TODO
    raise NotImplementedError

In [ ]:
test_z = np.linspace(-4, 4, 41)


def test_tanh():
    assert np.allclose(my_tanh(test_z), np.tanh(test_z)), "no coincide con np.tanh"


def test_tanh_derivative():
    assert np.allclose(my_tanh_derivative(test_z), numerical_derivative(np.tanh, test_z), atol=1e-6), \
        "no coincide con la derivada numérica"


check("my_tanh", test_tanh)
check("my_tanh_derivative", test_tanh_derivative)

## 7. Por qué importa la no linealidad

¿Hace falta la activación? ¿No alcanza con muchas capas lineales? Haz zoom a una foto, después otro zoom, después otro: el resultado es **un solo zoom**. Con las capas lineales pasa igual. Sin activación y sin sesgos:

$$W_3 \left( W_2 \left( W_1 x \right) \right) = \left( W_3 W_2 W_1 \right) x = W x$$

Cien capas lineales equivalen a una sola matriz. Compruébalo: tres capas contra la matriz $W$ que sale de multiplicarlas.

In [ ]:
lin_rng = np.random.default_rng(SEED)
W_a, W_b, W_c = lin_rng.normal(size=(5, 2)), lin_rng.normal(size=(5, 5)), lin_rng.normal(size=(1, 5))
points = lin_rng.normal(size=(1000, 2))

three_layers = (points @ W_a.T) @ W_b.T @ W_c.T
W_single = W_c @ W_b @ W_a                       # (1, 5) @ (5, 5) @ (5, 2) → (1, 2)
one_layer = points @ W_single.T
print("W_single:", W_single.shape, W_single.round(3))
print("¿tres capas = una sola?", np.allclose(three_layers, one_layer))

with_relu = relu(relu(points @ W_a.T) @ W_b.T) @ W_c.T
print("¿con ReLU también?     ", np.allclose(with_relu, one_layer))

### En datos: XOR con ruido

Llevemos XOR a datos de verdad: nubes de puntos alrededor de las cuatro esquinas, con la clase de su esquina. Entrenamos cuatro modelos más la línea base. Para entrenar usamos `MLPClassifier` de sklearn, una red como las de hoy con el patrón `fit/predict` de siempre; cómo encuentra los pesos es el tema de la semana 7. Todas las redes tienen **una capa oculta de 8 neuronas** y solo cambia su activación. `activation="identity"` significa ninguna: la salida de la suma pasa tal cual.

El `solver="lbfgs"` es un método de entrenamiento que funciona bien con pocos datos; los que usaremos con PyTorch llegan en la semana 10.

In [ ]:
data_rng = np.random.default_rng(SEED)
corners = data_rng.integers(0, 2, size=(400, 2))
X_xor = corners + data_rng.normal(0, 0.2, size=(400, 2))
y_xor = corners[:, 0] ^ corners[:, 1]
Xx_train, Xx_test, yx_train, yx_test = train_test_split(X_xor, y_xor, test_size=0.25, stratify=y_xor, random_state=SEED)


def make_mlp(activation, hidden=(8,)):
    return MLPClassifier(hidden, activation=activation, solver="lbfgs", max_iter=5000, random_state=SEED)


xor_models = {
    "línea base": DummyClassifier(strategy="most_frequent"),
    "regresión logística": LogisticRegression(),
    "red sin activación": make_mlp("identity"),
    "red con tanh": make_mlp("tanh"),
    "red con ReLU": make_mlp("relu"),
}
print(f"{'modelo':<22}{'entrenamiento':>14}{'prueba':>9}")
for name, model in xor_models.items():
    model.fit(Xx_train, yx_train)
    print(f"{name:<22}{model.score(Xx_train, yx_train):>14.3f}{model.score(Xx_test, yx_test):>9.3f}")

fig, axes = plt.subplots(1, 4, figsize=(12, 3.2), sharex=True, sharey=True)
for ax, name in zip(axes, list(xor_models)[1:]):
    plot_regions(ax, xor_models[name].predict, Xx_test, yx_test)
    ax.set(title=name, xlabel="$x_1$")
axes[0].set_ylabel("$x_2$")
fig.legend(*axes[0].get_legend_handles_labels(), loc="lower center", ncol=2, fontsize=9)
fig.subplots_adjust(bottom=0.27, wspace=0.08)
plt.show()

La red sin activación, con sus 8 neuronas, dibuja una recta igual que la regresión logística y se queda en el 50%, lo mismo que la línea base. **La misma red, con una activación, llega a más del 95%.** Eso es lo único que cambió.

### Una regla plegable

¿Cómo arma curvas una red con ReLU? Cada neurona oculta es una recta con **un doblez**, como una regla de carpintero. Sumando reglas dobladas en distintos lugares se puede seguir cualquier curva. Para verlo, construyamos a mano una aproximación de $\sin(x)$: una capa oculta donde cada neurona se dobla en un punto $t_i$, y la salida las suma con pesos $c_i$:

$$\hat{f}(x) = f(t_0) + \sum_i c_i \, \text{ReLU}(x - t_i)$$

Aquí no se entrena nada: $c_i$ es cuánto debe cambiar la pendiente en cada doblez para pasar por el siguiente punto de la curva.

In [ ]:
def relu_approximation(x, n_neurons, f=np.sin, low=-np.pi, high=np.pi):
    """Una capa oculta de n_neurons ReLU que sigue a f en [low, high], con pesos calculados a mano."""
    knots = np.linspace(low, high, n_neurons + 1)
    slopes = np.diff(f(knots)) / np.diff(knots)
    c = np.diff(slopes, prepend=0.0)                     # cuánto cambia la pendiente en cada doblez
    return f(knots[0]) + relu(x[:, None] - knots[:-1]) @ c


xs = np.linspace(-np.pi, np.pi, 500)
fig, axes = plt.subplots(1, 4, figsize=(12, 2.9), sharey=True)
for ax, n in zip(axes, (2, 4, 8, 16)):
    approx = relu_approximation(xs, n)
    ax.plot(xs, np.sin(xs), color="lightgray", linewidth=5, label="sin(x)")
    ax.plot(xs, approx, color="tab:blue", linewidth=2, label="la red")
    ax.set(title=f"{n} neuronas · error medio {np.abs(approx - np.sin(xs)).mean():.3f}", xlabel="x")
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.show()

Más neuronas, más dobleces, menos error. Este es el **teorema de aproximación universal** (*universal approximation theorem*, años 80 y 90): con suficientes neuronas, una sola capa oculta se acerca tanto como quieras a cualquier función continua en un intervalo. Sin no linealidad, ninguna cantidad de neuronas lo logra.

Dos advertencias. El teorema dice que los pesos **existen**, no cómo encontrarlos ni cuántas neuronas hacen falta. Y más neuronas también significan más flexibilidad para memorizar el ruido: el sobreajuste de la semana 5 sigue valiendo, y se detecta igual, con la validación.

### Todo junto: los dígitos

Volvamos a los dígitos, donde la red al azar acertaba como adivinar. La misma forma, $64 \to 32 \to 10$, ahora entrenada:

In [ ]:
digit_log = LogisticRegression(max_iter=2000).fit(X_dig_train, y_dig_train)
digit_mlp = MLPClassifier((32,), max_iter=1000, random_state=SEED).fit(X_dig_train, y_dig_train)

print(f"línea base (la clase más común):  accuracy {baseline_dig:.3f}")
print(f"red 64 → 32 → 10 con pesos al azar: accuracy {random_acc:.3f}")
print(f"regresión logística (una capa):   accuracy {digit_log.score(X_dig_test, y_dig_test):.3f}")
print(f"red 64 → 32 → 10 entrenada:       accuracy {digit_mlp.score(X_dig_test, y_dig_test):.3f}")
print("parámetros de la red:", sum(c.size for c in digit_mlp.coefs_) + sum(i.size for i in digit_mlp.intercepts_))

De adivinar a casi perfecto: esa es la diferencia entre los pesos al azar y los entrenados. Y una lección más: en estos dígitos pequeños y limpios, la regresión logística casi empata con la red. **La línea base y el modelo simple siempre van primero**: a veces no hace falta una red. Las redes marcan la diferencia con problemas más difíciles, como los que vienen en el resto del año.

## 8. Cierre

**Hoy vimos:**

- Perceptrón: suma con pesos más un escalón. Con dos entradas es una recta que separa el plano.
- Su regla de aprendizaje mueve los pesos solo cuando se equivoca, y encuentra la recta si existe.
- XOR no se separa con ninguna recta. Una capa oculta "dobla la hoja" y lo resuelve: es un MLP.
- La propagación hacia adelante es calcular capa por capa: $h = \text{ReLU}(W_1 x + b_1)$, $z = W_2 h + b_2$. Una capa de $a$ a $c$ tiene $a \cdot c + c$ parámetros.
- Activaciones: el escalón no da pendiente. Sigmoide y tanh se saturan. ReLU es la opción por defecto en las capas ocultas. La última capa depende del problema.
- Sin no linealidad, muchas capas equivalen a una. Con ella, una red puede acercarse a cualquier función.

**Para la semana 7:**

1. **Entrega al inicio de la clase**: **B3: Regresión lineal y logística desde cero** en NumPy, comparadas con sklearn sobre la misma partición. Se guarda en tu repositorio de entregas, en la carpeta `B3/`. Es el punto de control del bloque.
2. **Cuota semanal** en [la plataforma](https://ioai.artix.tech/).

Hoy pusimos los pesos a mano, y cuando hizo falta entrenar, `fit` lo hizo por dentro. La semana 7 abrimos esa caja: **backpropagation**, la regla de la cadena de la semana 4 aplicada a una red entera. Es lo que faltaba en 1969, y se popularizó en 1986.